In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [ ]:
image = cv2.imread('sar_1.jpg')
image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) 

In [ ]:
plt.imshow(image_gray, cmap="gray")

In [ ]:
# 1. Подберите парамтеры алгоритма разрастания регионов так, чтобы был выделен весь участок газона.

In [ ]:
import math
def homo_average(img, mask, point, T):
    av_val = img[mask > 0].sum() / np.count_nonzero(img[mask > 0])
                                                            
    if abs(av_val - img[point]) <= T:
        return True
    
    return False

In [ ]:
def region_growing(image, seed_point,homo_fun,r, T):
    mask = np.zeros(image_gray.shape, np.uint8)
    mask[seed_point] = 1
    count = 1
    while count > 0:
        count = 0
        local_mask = np.zeros(image_gray.shape, np.uint8)
        for i in range(r,image.shape[0] - r):
            for j in range(r,image.shape[1] - r):
                if mask[i,j]==0 and mask[i - r:i + r, j-r: j+r].sum() > 0:
                    if homo_fun(image, mask, (i,j), T):
                        local_mask[i,j] = 1
        count = np.count_nonzero(local_mask)
        print(count)
        mask += local_mask
        
    return mask*255
    

In [ ]:
seed_point = (250,250)
mask = region_growing(image_gray,seed_point,homo_average, 7, 20)

In [ ]:
plt.imshow(mask, cmap="gray")

In [ ]:
# 2. Реализуйте вычисление критерия однородности, отличного от представленного. Сравните результаты.

In [ ]:
def homo_median(img, mask, point, T):
    median = np.median(img[mask > 0])
    return abs(median - img[point]) <= T

mask_median = region_growing(image_gray, seed_point, homo_median, 7, 20)
plt.imshow(mask_median, cmap="gray")

In [ ]:
from skimage.transform import resize
import numpy as np
from skimage.metrics import structural_similarity, mean_squared_error

mask = resize(mask, image_gray.shape, order=0,
              preserve_range=True, anti_aliasing=False).astype(np.uint8)

mse_average = mean_squared_error(image_gray, mask)
mse_median  = mean_squared_error(image_gray, mask_median)
ssim_average, _ = structural_similarity(image_gray, mask,        data_range=255, full=True)
ssim_median,  _ = structural_similarity(image_gray, mask_median, data_range=255, full=True)

print(mse_average, ssim_average)
print(mse_median, ssim_median)

In [ ]:
# 3. Применить алгоритм сегментации watershed+distance transform для задачи подсчета пальмовых деревьев.

image = cv2.imread('palm_1.jpg')
image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
plt.imshow(image_gray, cmap="gray")

In [ ]:
ret, thresh = cv2.threshold(image_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

plt.imshow(thresh, cmap="gray")

In [ ]:
dist = cv2.distanceTransform(thresh, cv2.DIST_L2, 5) 
plt.imshow(dist, cmap="gray")

In [ ]:
ret, sure_fg = cv2.threshold(dist, 0.3 * dist.max(), 255, cv2.THRESH_BINARY)
plt.imshow(sure_fg, cmap="gray")

In [ ]:
sure_fg = sure_fg.astype(np.uint8)
ret, markers = cv2.connectedComponents(sure_fg)
plt.imshow(markers, cmap="gray")

In [ ]:
markers = cv2.watershed(image, markers)
plt.imshow(markers, cmap="gray")

In [ ]:
count = len(np.unique(markers)) - 1
print(f"Количество деревьев: {count}")